# NLP Preprocessing Engine

This notebook implements a robust NLP preprocessing pipeline designed to clean and transform textual data.

The pipeline handles:
- Text normalization
- Noise removal (URLs, emojis, numbers, etc.)
- Token filtering
- Statistical analysis


## Task 1: Conceptual Understanding

### 1. Difference between "Love" and "love"
In NLP, "Love" and "love" are treated as different tokens due to case sensitivity. This increases vocabulary size unnecessarily and may split semantic meaning. Converting all text to lowercase ensures both are treated as the same word, improving consistency and model performance.

---

### 2. What happens if stopwords are not removed?
If stopwords are not removed:
- Vocabulary size increases
- Noise in the data increases
- Model training becomes slower
- Important signals may be diluted

However, in some cases, keeping stopwords is beneficial for preserving meaning.

---

### 3. When removing stopwords is harmful

1. Sentiment Analysis  
Example: "I am not happy"  
Removing "not" changes the meaning completely.

2. Question Answering Systems  
Example: "What is the time?"  
Removing "what" removes the intent of the question.

---

### 4. Stemming vs Lemmatization

Stemming:
- Removes suffixes using simple rules
- May produce incorrect words  
Example: "running" → "runn"

Lemmatization:
- Uses vocabulary and grammar rules
- Produces meaningful base words  
Example: "running" → "run"

Conclusion:
Stemming is faster but less accurate, while lemmatization is slower but more precise.

In [3]:
import re

def preprocess_text(text):

    if not text or not isinstance(text, str):
        return [], ""


    text = re.sub(r"http\S+|www\S+|\S+@\S+", "", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)


    text = text.lower()


    text = re.sub(r"(.)\1{2,}", r"\1", text)


    text = re.sub(r"[^a-z\s]", "", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenization
    tokens = text.split()


    tokens = [t for t in tokens if len(t) > 2 or t in ["no", "not"]]


    cleaned_sentence = " ".join(tokens)

    return tokens, cleaned_sentence

In [5]:
samples = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product 😍😍",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

results = []

for text in samples:
    tokens, cleaned = preprocess_text(text)

    total_tokens = len(tokens)
    unique_tokens = len(set(tokens))
    avg_len = round(sum(len(t) for t in tokens) / total_tokens, 2) if total_tokens else 0

    results.append({
        "original": text,
        "tokens": tokens,
        "cleaned": cleaned,
        "total": total_tokens,
        "unique": unique_tokens,
        "avg_len": avg_len
    })

for r in results:
    print("\nOriginal Text:", r["original"])
    print("Cleaned Tokens:", r["tokens"])
    print("Cleaned Sentence:", r["cleaned"])
    print("Stats -> Total:", r["total"], "| Unique:", r["unique"], "| Avg Length:", r["avg_len"])


Original Text: Get 100% FREE access now!!!
Cleaned Tokens: ['get', 'free', 'access', 'now']
Cleaned Sentence: get free access now
Stats -> Total: 4 | Unique: 4 | Avg Length: 4.0

Original Text: I absolutely looooved this product 😍😍
Cleaned Tokens: ['absolutely', 'loved', 'this', 'product']
Cleaned Sentence: absolutely loved this product
Stats -> Total: 4 | Unique: 4 | Avg Length: 6.5

Original Text: Worst service ever... 0/10
Cleaned Tokens: ['worst', 'service', 'ever']
Cleaned Sentence: worst service ever
Stats -> Total: 3 | Unique: 3 | Avg Length: 5.33

Original Text: Call me at 9876543210
Cleaned Tokens: ['call']
Cleaned Sentence: call
Stats -> Total: 1 | Unique: 1 | Avg Length: 4.0

Original Text: This is THE best course!!!
Cleaned Tokens: ['this', 'the', 'best', 'course']
Cleaned Sentence: this the best course
Stats -> Total: 4 | Unique: 4 | Avg Length: 4.25

Original Text: Nooooo this is baaad!!!
Cleaned Tokens: ['no', 'this', 'bad']
Cleaned Sentence: no this bad
Stats -> Total:

In [ ]:
## Analysis

### Which sentence had the most noise?
"Win $$$ now!!! Limited offer!!!"
- Contains symbols, repeated punctuation, and spam patterns

### Which sentence retained the most meaningful tokens?
"I am not happy with this"
- Retains the word "not", which is critical for sentiment understanding

In [6]:
from collections import Counter

all_tokens = []

for r in results:
    all_tokens.extend(r["tokens"])

freq = Counter(all_tokens)

print("Top 10 Most Frequent Words:")
print(freq.most_common(10))

print("\nTop 5 Least Frequent Words:")
print(freq.most_common()[-5:])

Top 10 Most Frequent Words:
[('this', 4), ('now', 2), ('get', 1), ('free', 1), ('access', 1), ('absolutely', 1), ('loved', 1), ('product', 1), ('worst', 1), ('service', 1)]

Top 5 Least Frequent Words:
[('limited', 1), ('offer', 1), ('not', 1), ('happy', 1), ('with', 1)]


In [7]:
def full_pipeline(text_list):
    all_tokens = []
    clean_sentences = []

    for text in text_list:
        tokens, cleaned = preprocess_text(text)
        all_tokens.extend(tokens)
        clean_sentences.append(cleaned)

    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }

# Example usage
output = full_pipeline(samples)
print(output)

{'tokens': ['get', 'free', 'access', 'now', 'absolutely', 'loved', 'this', 'product', 'worst', 'service', 'ever', 'call', 'this', 'the', 'best', 'course', 'no', 'this', 'bad', 'got', 'win', 'now', 'limited', 'offer', 'not', 'happy', 'with', 'this'], 'clean_sentences': ['get free access now', 'absolutely loved this product', 'worst service ever', 'call', 'this the best course', 'no this bad', 'got', 'win now limited offer', 'not happy with this']}


## Error Handling

The preprocessing function handles edge cases:

- Empty string → returns empty output
- Only emojis → removed during cleaning
- Only numbers → removed completely
- Invalid input (None, non-string) → safely handled

This ensures robustness in real-world scenarios.